In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 17_bbox_padding_eval.ipynb
#
# segmentación previa mediante bounding box de
# personas con padding, como alternativa a la máscara
# píxel a píxel evaluada en el notebook 15.
#
# Motivación: la máscara exacta recortaba el arma cuando
# sobresalía del contorno corporal (58 TP→FN). Expandir el
# bbox de la persona con un margen (padding) debería mantener
# el arma dentro de la región de interés.
#
# Configs evaluadas (procesadas en el mismo bucle):
#   Config A   — baseline: frame completo (sin segmentación)
#   Config C10 — bbox persona + 10% padding
#   Config C20 — bbox persona + 20% padding
#   Config C30 — bbox persona + 30% padding
#
# En todos los casos:
#   - yolov8s-seg detecta personas
#   - Se expande su bbox con el padding correspondiente
#   - El área fuera de todos los bboxes expandidos va a negro
#   - El frame resultante se pasa a weapon_model
#
# Bloques:
#   BLOQUE A — mAP frame-level   (clips positivos)
#   BLOQUE B — Clasificación clip-level (positivos + negativos)
#   BLOQUE C — FP por categoría negativa
#   BLOQUE D — Guardado en Drive
#
# Outputs en OUT_DIR (mismo directorio que notebook 15):
#   results_config_{cfg}.txt
#   clip_results_{cfg}.csv
# ============================================================

import json
import os
import shutil
import csv
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from ultralytics import YOLO

print('✅ Imports OK')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Imports OK


In [ ]:
# ============================================================
# CONFIG
# ============================================================

POS_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt'
NEG_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt'
OUT_DIR  = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation'

WEAPON_DET_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'
SEG_WEIGHTS        = 'yolov8s-seg.pt'

# Inferencia
IMG_SIZE     = 640
CONF_WEAPON  = 0.25
IOU_NMS      = 0.7
CONF_SEG     = 0.4
FRAME_STRIDE = 1
MAX_SECONDS  = 15

# Umbral clip-level
DETECTION_THRESHOLD = 5

# IOU thresholds para mAP
IOU_THRESHOLDS = np.arange(0.5, 1.0, 0.05)

# Configs a evaluar: nombre -> padding fraction (0.0 = sin padding = Config A)
CONFIGS = {
    'A':   0.00,   # baseline: frame completo
    'C10': 0.10,   # bbox + 10%
    'C20': 0.20,   # bbox + 20%
    'C30': 0.30,   # bbox + 30%
}

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('✅ Config OK')
print(f'   Configs: {list(CONFIGS.keys())}')
print(f'   OUT_DIR: {OUT_DIR}')

✅ Config OK
   Configs: ['A', 'C10', 'C20', 'C30']
   OUT_DIR: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation


In [ ]:
# ============================================================
# CARGAR MODELOS
# ============================================================

LOCAL_WEAPON_WEIGHTS = '/content/weapon_best.pt'
if not os.path.exists(LOCAL_WEAPON_WEIGHTS):
    print('Copiando pesos del detector desde Drive...')
    shutil.copy2(WEAPON_DET_WEIGHTS, LOCAL_WEAPON_WEIGHTS)

weapon_model = YOLO(LOCAL_WEAPON_WEIGHTS)
seg_model    = YOLO(SEG_WEIGHTS)

print('✅ weapon_model:', LOCAL_WEAPON_WEIGHTS)
print('✅ seg_model:   ', SEG_WEIGHTS)

Copiando pesos del detector desde Drive...
✅ weapon_model: /content/weapon_best.pt
✅ seg_model:    yolov8s-seg.pt


In [ ]:
# ============================================================
# HELPERS
# ============================================================

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)


def load_gt_boxes(label_json_path):
    with open(label_json_path) as f:
        data = json.load(f)
    gt = defaultdict(list)
    for ann in data['annotations']:
        x, y, w, h = ann['bbox']
        gt[ann['image_id']].append((x, y, x + w, y + h))
    return gt


def build_bbox_mask(frame, seg_result, padding):
    """
    Construye una máscara binaria a partir de los bounding boxes
    de personas detectadas, expandidos con el padding indicado.

    Args:
      frame      : frame original (H, W, 3)
      seg_result : resultado de seg_model.predict()
      padding    : fracción de expansión del bbox (e.g. 0.20 = 20%)
                   Si padding == 0.0 se devuelve el frame original
                   (Config A, baseline)

    Retorna:
      masked_frame : frame con fondo a negro fuera de los bboxes
      n_persons    : número de personas detectadas
    """
    h, w = frame.shape[:2]

    # Config A: sin enmascarado
    if padding == 0.0:
        return frame.copy(), 0

    combined_mask = np.zeros((h, w), dtype=bool)
    n_persons = 0

    if seg_result.boxes is not None and len(seg_result.boxes) > 0:
        classes = seg_result.boxes.cls.cpu().numpy().astype(int)
        bboxes  = seg_result.boxes.xyxy.cpu().numpy()  # (N, 4) en coords de imagen

        for idx, cls_id in enumerate(classes):
            if cls_id != 0:   # solo personas
                continue

            x1, y1, x2, y2 = bboxes[idx]
            bw = x2 - x1
            bh = y2 - y1

            # Expandir bbox con padding
            pad_x = bw * padding
            pad_y = bh * padding
            x1p = int(max(0,   x1 - pad_x))
            y1p = int(max(0,   y1 - pad_y))
            x2p = int(min(w-1, x2 + pad_x))
            y2p = int(min(h-1, y2 + pad_y))

            combined_mask[y1p:y2p, x1p:x2p] = True
            n_persons += 1

    # Si no se detectó ninguna persona, usar el frame completo
    # (evita descartar frames donde la persona no fue segmentada)
    if n_persons == 0:
        return frame.copy(), 0

    masked_frame = np.zeros_like(frame)
    masked_frame[combined_mask] = frame[combined_mask]
    return masked_frame, n_persons


def predict_weapons(frame_input):
    r = weapon_model.predict(frame_input, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                              iou=IOU_NMS, verbose=False, device='cuda')[0]
    preds = []
    if r.boxes is not None and len(r.boxes) > 0:
        for b in r.boxes:
            x1, y1, x2, y2 = map(float, b.xyxy[0])
            preds.append((x1, y1, x2, y2, float(b.conf[0])))
    return preds


print('✅ Helpers OK')

✅ Helpers OK


In [ ]:
# ============================================================
# FUNCIÓN PRINCIPAL: procesar un vídeo en todas las configs
# Se llama al seg_model UNA sola vez por frame y se reutiliza
# el resultado para construir las 3 máscaras de padding.
# ============================================================

def run_video_all_configs(video_path):
    """
    Procesa el vídeo frame a frame y devuelve los resultados
    de todas las configs en una sola pasada.

    Retorna:
      frame_results : dict cfg -> list of (frame_idx, preds)
      max_frames    : int
    """
    cap      = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_f    = min(n_frames, int(MAX_SECONDS * fps))

    frame_results = {cfg: [] for cfg in CONFIGS}
    frame_i = 0

    while frame_i < max_f:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_i % FRAME_STRIDE == 0:
            # Ejecutar seg_model UNA vez por frame
            # (se reutiliza para todas las configs con padding > 0)
            seg_r = seg_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_SEG,
                                      classes=[0], verbose=False, device='cuda')[0]

            for cfg, pad in CONFIGS.items():
                frame_input, _ = build_bbox_mask(frame, seg_r, pad)
                preds = predict_weapons(frame_input)
                frame_results[cfg].append((frame_i + 1, preds))

        frame_i += 1

    cap.release()
    return frame_results, max_f


print('✅ run_video_all_configs() OK')

✅ run_video_all_configs() OK


In [ ]:
# ============================================================
# BLOQUE A — mAP a nivel de frame (clips positivos)
# ============================================================
print('\n' + '='*60)
print('BLOQUE A — mAP a nivel de frame')
print('='*60)

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]

all_tp   = {cfg: defaultdict(list) for cfg in CONFIGS}
all_fp   = {cfg: defaultdict(list) for cfg in CONFIGS}
total_gt = 0

for vp in pos_paths:
    if not vp.exists():
        print(f'  ❌ No existe: {vp}')
        continue

    label_path = vp.parent / 'label.json'
    if not label_path.exists():
        print(f'  ⚠️  Sin label.json: {vp.parent.name}')
        continue

    clip_id  = vp.parent.name
    local_in = f'/content/{clip_id}_eval.mp4'
    shutil.copy2(str(vp), local_in)

    gt_boxes  = load_gt_boxes(label_path)
    total_gt += sum(len(v) for v in gt_boxes.values())

    # Una sola pasada por vídeo, todas las configs
    frame_results, _ = run_video_all_configs(local_in)

    for cfg in CONFIGS:
        for (frame_idx, preds) in frame_results[cfg]:
            gts = gt_boxes.get(frame_idx, [])
            for iou_thr in IOU_THRESHOLDS:
                matched_gt = set()
                for (px1, py1, px2, py2, conf) in sorted(preds, key=lambda x: -x[4]):
                    best_iou, best_j = 0, -1
                    for j, gt in enumerate(gts):
                        if j in matched_gt:
                            continue
                        s = iou((px1, py1, px2, py2), gt)
                        if s > best_iou:
                            best_iou, best_j = s, j
                    if best_iou >= iou_thr and best_j >= 0:
                        all_tp[cfg][iou_thr].append(1)
                        all_fp[cfg][iou_thr].append(0)
                        matched_gt.add(best_j)
                    else:
                        all_tp[cfg][iou_thr].append(0)
                        all_fp[cfg][iou_thr].append(1)

    os.remove(local_in)
    print(f'  ✅ {clip_id}')

# Calcular métricas
map_metrics = {}
for cfg in CONFIGS:
    aps = {}
    for iou_thr in IOU_THRESHOLDS:
        tp = sum(all_tp[cfg][iou_thr])
        fp = sum(all_fp[cfg][iou_thr])
        fn = total_gt - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        aps[iou_thr] = {'precision': precision, 'recall': recall}
    map_metrics[cfg] = {
        'mAP50':   aps[0.5]['precision'],
        'mAP5095': float(np.mean([v['precision'] for v in aps.values()])),
        'prec50':  aps[0.5]['precision'],
        'rec50':   aps[0.5]['recall'],
    }

print(f'\n  Total GT boxes : {total_gt}')
header = ''.join(f'{cfg:>12}' for cfg in CONFIGS)
print(f'  {"":20} {header}')
print('  ' + '-' * (20 + 12 * len(CONFIGS)))
for key, label in [("mAP50","mAP@50"),("mAP5095","mAP@50:95"),
                   ("prec50","Precision@50"),("rec50","Recall@50")]:
    vals = ''.join(f'{map_metrics[cfg][key]:>12.4f}' for cfg in CONFIGS)
    print(f'  {label:20} {vals}')


BLOQUE A — mAP a nivel de frame
  ✅ PAH1_C1_P1_V1_HB_3
  ✅ PAH1_C1_P1_V1_HB_4
  ✅ PAH1_C1_P2_V1_HB_1
  ✅ PAH1_C1_P2_V1_HB_3
  ✅ PAH1_C1_P4_V1_HB_1
  ✅ PAH1_C1_P4_V1_HB_2
  ✅ PAH1_C2_P3_V1_HB_1
  ✅ PAH1_C2_P3_V1_HB_3
  ✅ PAH1_C2_P3_V2_HB_2
  ✅ PAH1_C2_P3_V2_HB_3
  ✅ PAH1_C2_P5_V1_HB_1
  ✅ PAH1_C2_P5_V1_HB_4
  ✅ PAH1_C2_P5_V2_HB_3
  ✅ PAH1_C2_P5_V2_HB_4
  ✅ PAH2_C1_P1_V1_HB_1
  ✅ PAH2_C1_P1_V1_HB_4
  ✅ PAH2_C1_P2_V1_HB_3
  ✅ PAH2_C1_P2_V1_HB_4
  ✅ PAH2_C1_P4_V1_HB_1
  ✅ PAH2_C1_P4_V1_HB_3
  ✅ PAH2_C2_P3_V1_HB_1
  ✅ PAH2_C2_P3_V1_HB_4
  ✅ PAH2_C2_P3_V2_HB_2
  ✅ PAH2_C2_P3_V2_HB_4
  ✅ PAH2_C2_P5_V1_HB_3
  ✅ PAH2_C2_P5_V1_HB_4
  ✅ PAH2_C2_P5_V2_HB_2
  ✅ PAH2_C2_P5_V2_HB_4
  ✅ PAH3_C1_P1_V1_HB_1
  ✅ PAH3_C1_P1_V1_HB_4
  ✅ PAH3_C1_P2_V1_HB_1
  ✅ PAH3_C1_P2_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_4
  ✅ PAH3_C2_P3_V1_HB_1
  ✅ PAH3_C2_P3_V1_HB_2
  ✅ PAH3_C2_P3_V2_HB_1
  ✅ PAH3_C2_P3_V2_HB_3
  ✅ PAH3_C2_P5_V1_HB_2
  ✅ PAH3_C2_P5_V1_HB_4
  ✅ PAH3_C2_P5_V2_HB_2
  ✅ PAH3_C2_P5_V2_HB_4
 

In [ ]:
# ============================================================
# BLOQUE B — Clasificación a nivel de clip
# ============================================================
print('\n' + '='*60)
print(f'BLOQUE B — Clasificación clip-level (umbral={DETECTION_THRESHOLD} frames)')
print('='*60)

neg_paths    = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]
clip_results = {cfg: [] for cfg in CONFIGS}

cat_labels = {
    'N1': 'Walking empty hands',  'N2': 'Jogging',
    'N3': 'Running',              'N4': 'Sneaking empty hands',
    'N5': 'Phone relaxed',        'N6': 'Phone looking',
    'N7': 'Phone both hands',     'N8': 'Phone recording 1h',
    'N9': 'Phone recording 2h',   'N10': 'Water bottle relaxed',
    'N11': 'Drinking',            'N12': 'Holding heavy object',
}


def classify_from_results(frame_results):
    frames_with_gun = sum(1 for (_, preds) in frame_results if len(preds) > 0)
    return (1 if frames_with_gun >= DETECTION_THRESHOLD else 0), frames_with_gun


def process_split(paths, true_label, split_name):
    cfg_names = list(CONFIGS.keys())
    print(f'\n  Procesando {split_name} ({len(paths)} clips)...')
    print(f'  {"Clip":<42}  ' + '  '.join(f'{c:>6}' for c in cfg_names))
    print('  ' + '-' * (42 + 10 * len(cfg_names)))

    for vp in paths:
        if not vp.exists():
            continue
        clip_id  = vp.parent.name
        category = clip_id.split('_')[0] if true_label == 0 else 'POS'
        local_in = '/content/tmp_cls.mp4'
        shutil.copy2(str(vp), local_in)

        frame_results, n_total = run_video_all_configs(local_in)
        os.remove(local_in)

        labels_row = []
        for cfg in CONFIGS:
            pred, n_det = classify_from_results(frame_results[cfg])
            clip_results[cfg].append({
                'clip':         clip_id,
                'true':         true_label,
                'pred':         pred,
                'det_frames':   n_det,
                'total_frames': n_total,
                'category':     category,
            })
            if true_label == 1:
                labels_row.append('TP' if pred == 1 else 'FN')
            else:
                labels_row.append('FP' if pred == 1 else 'TN')

        print(f'  {clip_id:<42}  ' + '  '.join(f'{l:>6}' for l in labels_row))


process_split(pos_paths, true_label=1, split_name='positivos')
process_split(neg_paths, true_label=0, split_name='negativos')


def compute_metrics(results):
    y_true = np.array([r['true'] for r in results])
    y_pred = np.array([r['pred'] for r in results])
    TP = int(((y_true==1) & (y_pred==1)).sum())
    TN = int(((y_true==0) & (y_pred==0)).sum())
    FP = int(((y_true==0) & (y_pred==1)).sum())
    FN = int(((y_true==1) & (y_pred==0)).sum())
    acc  = (TP+TN) / len(y_true)
    prec = TP / (TP+FP) if (TP+FP) > 0 else 0
    rec  = TP / (TP+FN) if (TP+FN) > 0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    return {'TP':TP,'TN':TN,'FP':FP,'FN':FN,
            'acc':acc,'prec':prec,'rec':rec,'f1':f1}


m = {cfg: compute_metrics(clip_results[cfg]) for cfg in CONFIGS}

print('\n  --- Métricas globales clip-level ---')
header = ''.join(f'{cfg:>12}' for cfg in CONFIGS)
print(f'  {"":20} {header}')
print('  ' + '-' * (20 + 12 * len(CONFIGS)))
for key, label in [("acc","Accuracy"),("prec","Precision"),
                   ("rec","Recall"),("f1","F1")]:
    vals = ''.join(f'{m[cfg][key]:>12.4f}' for cfg in CONFIGS)
    print(f'  {label:20} {vals}')
for key, label in [("TP","TP"),("TN","TN"),("FP","FP"),("FN","FN")]:
    vals = ''.join(f'{m[cfg][key]:>12}' for cfg in CONFIGS)
    print(f'  {label:20} {vals}')


BLOQUE B — Clasificación clip-level (umbral=5 frames)

  Procesando positivos (140 clips)...
  Clip                                             A     C10     C20     C30
  ----------------------------------------------------------------------------------
  PAH1_C1_P1_V1_HB_3                              TP      TP      TP      TP
  PAH1_C1_P1_V1_HB_4                              TP      TP      TP      TP
  PAH1_C1_P2_V1_HB_1                              TP      TP      TP      TP
  PAH1_C1_P2_V1_HB_3                              TP      TP      TP      TP
  PAH1_C1_P4_V1_HB_1                              TP      TP      TP      TP
  PAH1_C1_P4_V1_HB_2                              TP      TP      TP      TP
  PAH1_C2_P3_V1_HB_1                              TP      TP      TP      TP
  PAH1_C2_P3_V1_HB_3                              TP      TP      TP      TP
  PAH1_C2_P3_V2_HB_2                              TP      TP      TP      TP
  PAH1_C2_P3_V2_HB_3                              T

In [ ]:
# ============================================================
# BLOQUE C — FP por categoría negativa
# ============================================================
print('\n' + '='*60)
print('BLOQUE C — Falsos positivos por categoría negativa')
print('='*60)

cat_stats = {cfg: defaultdict(lambda: {'total': 0, 'fp': 0}) for cfg in CONFIGS}

for cfg in CONFIGS:
    for r in clip_results[cfg]:
        if r['true'] != 0:
            continue
        cat = r['category']
        cat_stats[cfg][cat]['total'] += 1
        if r['pred'] == 1:
            cat_stats[cfg][cat]['fp'] += 1

cfg_list = list(CONFIGS.keys())
header   = ''.join(f'  {cfg:>5} {"":>6}' for cfg in cfg_list)
print(f"\n  {'Cat':<5} {'Descripción':<24}" + header)
print('  ' + '-' * (29 + 14 * len(cfg_list)))

for cat in sorted(cat_stats['A'], key=lambda x: int(x[1:])):
    desc = cat_labels.get(cat, cat)
    row  = f'  {cat:<5} {desc:<24}'
    for cfg in cfg_list:
        s   = cat_stats[cfg][cat]
        tot = s['total']
        pct = s['fp'] / tot * 100 if tot > 0 else 0
        row += f"  {s['fp']:>2}/{tot:<2} {pct:>5.1f}%"
    print(row)


BLOQUE C — Falsos positivos por categoría negativa

  Cat   Descripción                   A           C10           C20           C30       
  -------------------------------------------------------------------------------------
  N1    Walking empty hands        3/9   33.3%   2/9   22.2%   4/9   44.4%   4/9   44.4%
  N2    Jogging                    3/9   33.3%   2/9   22.2%   2/9   22.2%   1/9   11.1%
  N3    Running                    0/8    0.0%   0/8    0.0%   0/8    0.0%   0/8    0.0%
  N4    Sneaking empty hands       2/7   28.6%   1/7   14.3%   1/7   14.3%   1/7   14.3%
  N5    Phone relaxed              2/10  20.0%   6/10  60.0%   4/10  40.0%   4/10  40.0%
  N6    Phone looking              8/16  50.0%   4/16  25.0%   6/16  37.5%   7/16  43.8%
  N7    Phone both hands           4/7   57.1%   4/7   57.1%   5/7   71.4%   5/7   71.4%
  N8    Phone recording 1h         9/15  60.0%  11/15  73.3%  11/15  73.3%  11/15  73.3%
  N9    Phone recording 2h         7/9   77.8%   6/9   66.

In [ ]:
# ============================================================
# BLOQUE D — Guardar resultados en Drive
# ============================================================
print('\n' + '='*60)
print('BLOQUE D — Guardando resultados')
print('='*60)

cfg_name_map = {
    'A':   'sin_seg',
    'C10': 'bbox_pad10',
    'C20': 'bbox_pad20',
    'C30': 'bbox_pad30',
}

for cfg in CONFIGS:
    label = cfg_name_map[cfg]

    # TXT resumen
    txt_path = Path(OUT_DIR) / f'results_config_{cfg}_{label}.txt'
    with open(txt_path, 'w') as f:
        f.write(f'=== CONFIG {cfg} — {label.upper()} ===\n\n')
        f.write(f'WEAPON_MODEL: {WEAPON_DET_WEIGHTS}\n')
        pad = CONFIGS[cfg]
        if pad == 0.0:
            f.write('Estrategia: frame completo (baseline)\n')
        else:
            f.write(f'Estrategia: bbox persona + {int(pad*100)}% padding\n')
            f.write(f'SEG_MODEL: {SEG_WEIGHTS} | CONF_SEG={CONF_SEG}\n')
        f.write(f'CONF_WEAPON={CONF_WEAPON} | IOU_NMS={IOU_NMS} | '
                f'DETECTION_THRESHOLD={DETECTION_THRESHOLD}\n\n')

        f.write('--- BLOQUE A: mAP frame-level ---\n')
        f.write(f'Total GT boxes : {total_gt}\n')
        for key, lbl in [("mAP50","mAP@50"),("mAP5095","mAP@50:95"),
                         ("prec50","Precision@50"),("rec50","Recall@50")]:
            f.write(f'{lbl:15}: {map_metrics[cfg][key]:.4f}\n')
        f.write('\n')

        f.write('--- BLOQUE B: clip-level ---\n')
        for key, lbl in [("acc","Accuracy"),("prec","Precision"),
                         ("rec","Recall"),("f1","F1")]:
            f.write(f'{lbl:15}: {m[cfg][key]:.4f}\n')
        f.write(f'TP={m[cfg]["TP"]} TN={m[cfg]["TN"]} '
                f'FP={m[cfg]["FP"]} FN={m[cfg]["FN"]}\n\n')

        f.write('--- BLOQUE C: FP por categoría ---\n')
        for cat in sorted(cat_stats[cfg], key=lambda x: int(x[1:])):
            s   = cat_stats[cfg][cat]
            tot = s['total']
            pct = s['fp'] / tot * 100 if tot > 0 else 0
            f.write(f'  {cat}: {s["fp"]}/{tot} ({pct:.1f}%)\n')

    print(f'  ✅ {txt_path}')

    # CSV detallado
    csv_path = Path(OUT_DIR) / f'clip_results_{cfg}_{label}.csv'
    with open(csv_path, 'w', newline='') as f:
        fieldnames = ['clip','true','category','det_frames','total_frames','pred']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(clip_results[cfg])
    print(f'  ✅ {csv_path}')

print('\n✅ Evaluación completa.')


BLOQUE D — Guardando resultados
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/results_config_A_sin_seg.txt
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/clip_results_A_sin_seg.csv
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/results_config_C10_bbox_pad10.txt
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/clip_results_C10_bbox_pad10.csv
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/results_config_C20_bbox_pad20.txt
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/clip_results_C20_bbox_pad20.csv
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/results_config_C30_bbox_pad30.txt
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/clip_results_C30_bbox_pad30.csv

✅ Evaluación completa.
